# 01 – Data Ingestion
The goals are:
- Load raw CSV files without modification
- Persist them into a SQLite database
- Validate table creation and row counts

In [5]:

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import sqlite3

## Imports and Environment Setup

This cell imports the core libraries required for:
- Data base connection
- Creating and interacting with a SQLite database
- Executing SQL queries from Python

In [2]:
conn = create_engine("sqlite:///instacart.db")
conn= sqlite3.connect("instacart.db")

## Load Raw CSV Files into SQLite

In this step:
- Each CSV file is read into a pandas DataFrame
- The DataFrame is written directly to SQLite
- No transformations are applied

This preserves the raw dataset exactly as provided.


In [3]:
files = {
    "orders": r"C:\Users\shiva\Downloads\archive (5)\orders.csv",
    "products": r"C:\Users\shiva\Downloads\archive (5)\products.csv",
    "order_products_prior": r"C:\Users\shiva\Downloads\archive (5)\order_products__prior.csv",
    "order_products_train": r"C:\Users\shiva\Downloads\archive (5)\order_products__train.csv",
    "aisles": r"C:\Users\shiva\Downloads\archive (5)\aisles.csv",
    "departments": r"C:\Users\shiva\Downloads\archive (5)\departments.csv"
}

for table_name, file_path in files.items():
    df = pd.read_csv(file_path)
    df.to_sql(table_name,conn, if_exists="replace", index=False)
    
    print(f"{table_name}: {df.shape}")

orders: (3421083, 7)
products: (49688, 4)
order_products_prior: (32434489, 4)
order_products_train: (1384617, 4)
aisles: (134, 2)
departments: (21, 2)


## Verify Table Creation in SQLite

This query checks:
- Whether all expected tables were created
- The SQL schema associated with each table

In [4]:
pd.read_sql("""
SELECT name, sql
FROM sqlite_master
WHERE type='table';
""", conn)

,name,sql
0,orders,"CREATE TABLE ""orders"" (\n""order_id"" INTEGER,\n..."
1,products,"CREATE TABLE ""products"" (\n""product_id"" INTEGE..."
2,order_products_prior,"CREATE TABLE ""order_products_prior"" (\n""order_..."
3,order_products_train,"CREATE TABLE ""order_products_train"" (\n""order_..."
4,aisles,"CREATE TABLE ""aisles"" (\n""aisle_id"" INTEGER,\n..."
5,departments,"CREATE TABLE ""departments"" (\n""department_id"" ..."


## Validate Row Counts

We compare row counts in SQLite against expectations.

This ensures:
- No partial ingestion
- No silent truncation
- No missing tables


In [5]:
tables = [
    "orders",
    "order_products_prior",
    "order_products_train",
    "products",
    "aisles",
    "departments"
]

for t in tables:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {t}",conn)
    print(f"{t}: {count.iloc[0]['cnt']}")


orders: 3421083
order_products_prior: 32434489
order_products_train: 1384617
products: 49688
aisles: 134
departments: 21


## Summary

At this point:
- All Instacart CSV files are successfully loaded into SQLite
- Table schemas and row counts are validated
- The database is ready for downstream SQL-based analysis
